# 01 — Нормализация текста-шаблона
## Цель
Загрузить FB2-файл романа «Бесы», извлечь текст основного произведения, нормализовать и подготовить единую координатную строку для дальнейшего сопоставления с аудио и страницами.

## Структура FB2
- **Body 0 / Section «Бесы»** — основной текст: Части 1–3, 23 главы, ~1M символов
- **Body 0 / Section «Приложение» / «Глава девятая. У Тихона»** — соответствует аудиотреку 309
- **Body 0 / Section «Комментарии»** — Endnotes, НЕ включаем (нет аудио)
- **Body 1, 2** — Ссылки на комментарии, НЕ включаем

## Выход
- `normalized_text` — строка всего текста в нижнем регистре
- `char_index_to_source` — dict: позиция символа → (часть, глава, параграф)
- `section_boundaries` — границы частей/глав для диагностики

In [1]:
import pickle, re, os
from pathlib import Path
from lxml import etree

# Корень проекта — для переносимости определяем относительно этого файла
PROJECT_ROOT = Path(os.environ.get(
    "SPARK_ROOT",
    "/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit"
))

In [2]:
# Пути
DATA_DIR = PROJECT_ROOT / "data/besy"
OUTPUT_DIR = PROJECT_ROOT / "outputs/besy/run_01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FB2_PATH = DATA_DIR / "text/besy.fb2"
print(f"FB2: {FB2_PATH}")
print(f"Exists: {FB2_PATH.exists()}")

FB2: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/data/besy/text/besy.fb2
Exists: True


In [3]:
tree = etree.parse(str(FB2_PATH))
root = tree.getroot()
NS = "http://www.gribuser.ru/xml/fictionbook/2.0"

def tag(name):
    return f"{{{NS}}}{name}"

In [4]:
def extract_paragraphs(section_element):
    """Извлечь все <p> внутри секции (прямые и вложенные)."""
    return section_element.findall(f".//{tag('p')}")

def get_title(section_element):
    """Извлечь заголовок секции."""
    title_el = section_element.find(tag('title'))
    if title_el is not None:
        title_p = title_el.find(tag('p'))
        if title_p is not None and title_p.text:
            return title_p.text.strip()
    return ""

In [5]:
# Находим тело с основным текстом
bodies = root.findall(tag('body'))
body0 = bodies[0]

# Секции верхнего уровня
top_sections = body0.findall(tag('section'))
print("Секции Body 0:")
for s in top_sections:
    print(f"  [{get_title(s)}] — вложенных секций: {len(s.findall(tag('section')))})")

Секции Body 0:
  [Бесы] — вложенных секций: 4)
  [Приложение] — вложенных секций: 2)
  [Комментарии] — вложенных секций: 0)


In [6]:
# Собираем основной текст: секция "Бесы" + "Приложение/У Тихона"
# Пропускаем "Комментарии" (endnotes) и "Зависть" (нет аудио)

besy_section = top_sections[0]  # "Бесы"
appendix_section = top_sections[1]  # "Приложение"

print("=== Структура романа «Бесы» ===")
parts = besy_section.findall(tag('section'))
for part in parts:
    part_title = get_title(part)
    chapters = part.findall(tag('section'))
    if chapters:
        p_count = sum(len(extract_paragraphs(ch)) for ch in chapters)
        print(f"  {part_title}: {len(chapters)} глав, {p_count} параграфов")
    else:
        print(f"  {part_title}: 0 глав (пустая секция)")

# Приложение
appendix_parts = appendix_section.findall(tag('section'))
for ap in appendix_parts:
    ap_title = get_title(ap)
    p_count = len(extract_paragraphs(ap))
    print(f"  [Приложение] {ap_title}: {p_count} параграфов")

=== Структура романа «Бесы» ===
  : 0 глав (пустая секция)
  Часть первая: 5 глав, 1310 параграфов
  Часть вторая: 10 глав, 2028 параграфов
  Часть третья: 8 глав, 1687 параграфов
  [Приложение] Глава девятая. У Тихона: 260 параграфов
  [Приложение] Зависть: 97 параграфов


In [7]:
def collect_text(section_element):
    """Рекурсивно собрать текст из секции.
    Возвращает список (paragraph_text, source_info).
    """
    results = []
    
    # Прямые параграфы этой секции
    paragraphs = section_element.findall(tag('p'))
    source = get_title(section_element)
    for p in paragraphs:
        text = (p.text or "").strip()
        if text:
            results.append((text, source))
    
    # Рекурсивно во вложенные секции
    for child in section_element.findall(tag('section')):
        results.extend(collect_text(child))
    
    return results

In [8]:
# Собираем весь текст романа (Части 1-3)
novel_paragraphs = []

# Сначала вступление (пустая секция) — проверяем прямой текст
for i, part in enumerate(parts):
    part_title = get_title(part)
    chapters = part.findall(tag('section'))
    
    if not chapters:
        # Пустая секция (intro) — берём прямые параграфы
        for p in part.findall(tag('p')):
            text = (p.text or "").strip()
            if text:
                novel_paragraphs.append((text, part_title or "Вступление"))
    else:
        for ch in chapters:
            ch_title = get_title(ch)
            source = f"{part_title} / {ch_title}"
            for p in ch.findall(tag('p')):
                text = (p.text or "").strip()
                if text:
                    novel_paragraphs.append((text, source))

print(f"Собрано параграфов из романа: {len(novel_paragraphs)}")

Собрано параграфов из романа: 4968


In [9]:
# Добавляем "У Тихона" из Приложения (соответствует аудиотреку 309)
tihon_paragraphs = []
for ap in appendix_parts:
    ap_title = get_title(ap)
    if "Тихон" in ap_title:
        for p in ap.findall(tag('p')):
            text = (p.text or "").strip()
            if text:
                tihon_paragraphs.append((text, f"Приложение / {ap_title}"))
        print(f"Добавлено из «{ap_title}»: {len(tihon_paragraphs)} параграфов")
        break

all_paragraphs = novel_paragraphs + tihon_paragraphs
print(f"Всего параграфов: {len(all_paragraphs)}")

Добавлено из «Глава девятая. У Тихона»: 259 параграфов
Всего параграфов: 5227


In [10]:
def normalize_text(text):
    """Нормализовать текст параграфа для сопоставления."""
    text = text.lower()
    text = re.sub(r'[^\w\s\-.,!?;:"\'Ѐ-ӿ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Нормализуем параграфы
normalized_paragraphs = []
for raw_text, source in all_paragraphs:
    norm_text = normalize_text(raw_text)
    if norm_text:
        normalized_paragraphs.append((norm_text, source))

# Строим итоговую строку и индекс
char_index_to_source = {}
section_boundaries = []
char_pos = 0
current_section = None
section_start = 0

for i, (norm_text, source) in enumerate(normalized_paragraphs):
    # Границы секций
    if source != current_section:
        if current_section is not None:
            section_boundaries.append((section_start, char_pos - 1, current_section))
        current_section = source
        section_start = char_pos

    # Символы параграфа
    for ch in norm_text:
        char_index_to_source[char_pos] = source
        char_pos += 1

    # Разделитель \n между параграфами (но не после последнего)
    if i < len(normalized_paragraphs) - 1:
        char_index_to_source[char_pos] = source
        char_pos += 1

# Последняя секция
if current_section is not None:
    section_boundaries.append((section_start, char_pos - 1, current_section))

# Итоговая строка
normalized_text = "\n".join(p[0] for p in normalized_paragraphs)

assert len(normalized_text) == char_pos, \
    f"Расхождение: текст={len(normalized_text)}, индекс={char_pos}"

print(f"Длина нормализованного текста: {len(normalized_text):,} символов")
print(f"Количество параграфов: {len(normalized_paragraphs)}")
print(f"Записей в индексе: {len(char_index_to_source):,}")
print(f"Совпадение: текст == индекс ✓")

Длина нормализованного текста: 1,086,475 символов
Количество параграфов: 5225
Записей в индексе: 1,086,475
Совпадение: текст == индекс ✓


In [11]:
# Примеры первых и последних параграфов
print("=== Первые 3 параграфа ===")
for i, (text, source) in enumerate(all_paragraphs[:3]):
    norm = normalize_text(text)
    if norm:
        print(f"[{source}] {norm[:150]}...")

print("\n=== Последние 3 параграфа ===")
for i, (text, source) in enumerate(all_paragraphs[-3:]):
    norm = normalize_text(text)
    if norm:
        print(f"[{source}] {norm[:150]}...")

=== Первые 3 параграфа ===
[Часть первая / Глава первая] приступая к описанию недавних и столь странных событий, происшедших в нашем, доселе ничем не отличавшемся городе, я принужден, по неумению моему, нача...
[Часть первая / Глава первая] скажу прямо: степан трофимович постоянно играл между нами некоторую особую и, так сказать, гражданскую роль и любил эту роль до страсти, так даже, что...
[Часть первая / Глава первая] я даже так думаю, что под конец его. все и везде позабыли; но уже никак ведь нельзя сказать, что и прежде совсем не знали. бесспорно, что и он некотор...

=== Последние 3 параграфа ===
[Приложение / Глава девятая. У Тихона] нет, не после обнародования, а еще до обнародования листков, за день, за час, может быть, до великого шага, вы броситесь в новое преступление как в ис...
[Приложение / Глава девятая. У Тихона] ставрогин даже задрожал от гнева и почти от испуга....
[Приложение / Глава девятая. У Тихона] проклятый психолог! оборвал он вдруг в бешенстве и, не оглядывая

In [12]:
# Статистика по частям
print("=== Распределение текста по частям ===")
for start, end, title in section_boundaries:
    length = end - start + 1
    pct = length / len(normalized_text) * 100
    # Показать фрагмент текста для проверки
    snippet = normalized_text[start:start+80]
    print(f"{title:40s} | {start:>8,}–{end:>8,} | {length:>8,} симв | {pct:5.1f}% | \"{snippet}…\"")

=== Распределение текста по частям ===
Часть первая / Глава первая              |        0–  35,793 |   35,794 симв |   3.3% | "приступая к описанию недавних и столь странных событий, происшедших в нашем, дос…"
Часть первая / Глава вторая              |   35,794– 103,220 |   67,427 симв |   6.2% | "на земле существовало еще одно лицо, к которому варвара петровна была привязана …"
Часть первая / Глава третья              |  103,221– 174,199 |   70,979 симв |   6.5% | "прошло с неделю, и дело начало несколько раздвигаться.
замечу вскользь, что в эт…"
Часть первая / Глава четвертая           |  174,200– 231,845 |   57,646 симв |   5.3% | "шатов не заупрямился и, по записке моей, явился в полдень к лизавете николаевне.…"
Часть первая / Глава пятая               |  231,846– 314,151 |   82,306 симв |   7.6% | "варвара петровна позвонила в колокольчик и бросилась в кресла у окна.
сядьте зде…"
Часть вторая / Глава первая              |  314,152– 394,372 |   80,221 симв |   7.4% | "прошло восем

In [13]:
# Сохраняем результаты
outputs = {
    "normalized_text": normalized_text,
    "char_index_to_source": char_index_to_source,
    "section_boundaries": section_boundaries,
    "paragraph_count": len(normalized_paragraphs),
    "source_file": str(FB2_PATH),
}

with open(OUTPUT_DIR / "normalized_text.pkl", "wb") as f:
    pickle.dump(outputs, f)

# Также сохраняем нормализованный текст как TXT для ручного просмотра
with open(OUTPUT_DIR / "normalized_text.txt", "w", encoding="utf-8") as f:
    f.write(normalized_text)

print(f"Сохранено в {OUTPUT_DIR}:")
print(f"  - normalized_text.pkl ({len(normalized_text):,} симв)")
print(f"  - normalized_text.txt (для ручной проверки)")

Сохранено в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01:
  - normalized_text.pkl (1,086,475 симв)
  - normalized_text.txt (для ручной проверки)


## Проверка

1. Открыть `outputs/besy/run_01/normalized_text.txt` в текстовом редакторе
2. Взять случайный фрагмент текста (например, середину главы)
3. Открыть оригинальный FB2 и найти этот же фрагмент
4. Убедиться, что нормализация не исказила слова и не нарушила порядок
5. Проверить, что «У Тихона» присутствует в конце текста